# B2.4 · Deduplication and contextual verification

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.3 · Vulnerability auditing: three generations of SAST](https://spbreed.github.io/cyber-commons/lessons/B2.3.html)**.

| | |
|---|---|
| Tools used | OpenGrep, tree-sitter, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Deduplicate findings across three analysis tracks, then verify each against the AST and drop the ones that reference code that is not there.

**Why a security engineer needs it.** Parallel analysis tracks report the same bug three times, and some of those bugs do not exist. The control it builds is: stages 8–9: consolidate overlapping findings, then cross-reference each one against syntax and imports to weed out hallucinations.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Three analysers found the same defect and reported it four times, in three vocabularies, at two severities. A queue that inflates by 3x is not a queue — it is a landfill with a ticket number.

> **At CyberTravels.** Three analysers found the same booking-handler defect four times. The queue Alex will actually read is the deduplicated one.

## 2 · The framework

```
   raw findings                        after dedup + context
   +---------------------------+       +--------------------+
   | analyser A: SQLI in q()   |       | SQLI in q()        |
   | analyser B: CWE-89 q()    | ----> |   3 analysers      |
   | analyser C: taint -> q()  |       |   1 ticket         |
   | analyser A: SQLI in q()   |       +--------------------+
   +---------------------------+
        3x inflation                   the queue a human will read
```

Stage 7 ran several analysers in parallel. That produces two problems this stage
exists to solve, and they are different problems.

**Stage 8 — Deduplication.** Three analysers find the same bug and report it
three times. Worse, they report it at slightly different line numbers with
different CWE labels, so naive matching does not collapse them. An engineer who
sees the same bug three times stops trusting the count.

**Stage 9 — Contextual verification.** Cross-reference each finding against the
actual syntax and imports to weed out hallucinations. This is the cheapest,
highest-yield filter in the whole pipeline, because a model finding that
references a function that does not exist, or a module that was never imported,
is *provably* wrong — no judgement required.

The order matters: deduplicate first, then verify, or you spend verification
effort on three copies of the same claim.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Stage 8 — deduplicate on the defect, not the report

Two findings are the same defect if they name the same sink in the same function, even at different lines and under different CWE labels. Cluster on that, and keep the *best-evidenced* member.

## 4 · Stage 9 — contextual verification against the real syntax

Now check each surviving claim against the code. Three checks, all mechanical, none requiring judgement.

## 5 · The stage, as a skill

Seven raw findings, four defects. The skill normalises the CWE aliases, keys each finding by its enclosing function rather than a line number, and then rejects the survivors whose symbols are not in the file — because a finding about `os.system` in a file that never imports `os` should die here rather than in a maintainer's inbox.

### The skill — [`skills/appsec/finding-dedup-and-verification/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/finding-dedup-and-verification/SKILL.md)

```yaml
name: finding-dedup-and-verification
description: >-
  Collapse raw findings from several tools into distinct defects across CWE
  aliases and duplicate reports, then reject the ones whose symbols do not
  appear in the file. Use when a pipeline produces more findings than defects,
  or when a scanner reports something that is not in the code.
allowed-tools: Read, Grep, Glob
```

# Findings are not defects, and some of them are not real

A pipeline running three analyses over one file produces findings that overlap
in two different ways — the same defect reported under an alias, and the same
defect reported by three tools — and, if a model is one of the tools, findings
about symbols that do not exist. Both have to be resolved before a human sees
the list, or the human resolves them by ignoring the list.

## When to use this

Between analysis and triage, in any pipeline with more than one analyser, and
always when a model is one of them.

## Procedure

**1 — Normalise the CWE.** Maintain an alias map — CWE-943 is CWE-89 for this
purpose — and normalise before comparing. Aliases are why the same defect
appears twice with different identifiers.

**2 — Key each finding by defect, not by report.** File, enclosing function,
normalised CWE. Line numbers move; the enclosing function does not, and it is
what makes two reports of one defect collapse.

**3 — When duplicates disagree, keep the one with the best evidence.** A taint
result carries a path; a grep hit carries a line; a model result carries a
sentence. Prefer in that order, and record which tools agreed — agreement is
useful signal even though it is not proof.

**4 — Verify each survivor against the source.** Does the symbol it names appear
in the file? Is the module it blames imported? A finding about `os.system` in a
file that never imports `os` is a hallucination, and it is rejected here rather
than in a maintainer's inbox.

**5 — Report both counts and the rejects.** Raw findings, distinct defects, and
the rejected list with the reason. Discarding hallucinations silently loses the
measurement of how often the analyser produces them.

## Output contract

```json
{
  "raw": 0,
  "defects": [{"key": "str", "file": "str", "function": "str", "cwe": "str",
               "reported_by": ["str"], "kept_from": "str"}],
  "rejected": [{"finding": "str", "reason": "symbol absent|module not imported"}],
  "counts": {"raw": 0, "distinct": 0, "rejected": 0}
}
```

## Failure modes

- **Keying on the line number.** One edit and the same defect is two.
- **Merging by CWE alone.** Two real SQL injections in one file are two defects.
- **Dropping hallucinations without counting them.** That count is how you know
  whether to keep the analyser.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/finding-dedup-and-verification/scripts/finding_dedup_and_verification.py
SCRIPT = "skills/appsec/finding-dedup-and-verification/scripts/finding_dedup_and_verification.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Seven raw findings collapse to four distinct defects, with the CWE-943 alias merging into CWE-89 and the taint result kept over grep and model duplicates. Contextual verification then rejects the hallucinated `DB_PASSWORD` and `os.system` findings because neither symbol appears in the file and `os` is never imported, leaving the real SQL injection.

## Your turn

Add a fourth verification check: does the CWE class match the sink type? A CWE-22 finding on a `conn.execute` call is provably mislabelled, and that check costs nothing to run.

---

**Next → [B2.5 · Feasibility filtering and reachability](https://spbreed.github.io/cyber-commons/lessons/B2.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*